# Profile PyHEARTS

Measure end-to-end runtime and separate the morphology core from the
record-level T stage.

Install optional simulation support first: `python -m pip install -e ".[sim]"`.

In [ ]:
from __future__ import annotations

import cProfile
import io
import pstats
import time

import neurokit2 as nk
import numpy as np
import pandas as pd

import pyhearts
from pyhearts import PyHEARTS
import pyhearts.core.analyzer as analyzer_mod

FS = 500.0
DURATION_S = 30
ECG = np.asarray(
    nk.ecg_simulate(
        duration=DURATION_S,
        sampling_rate=int(FS),
        heart_rate=70,
        random_state=42,
    ),
    dtype=float,
)

print("package:", pyhearts.__version__, pyhearts.__file__)
print(f"signal: {len(ECG)} samples ({DURATION_S} s @ {FS:g} Hz)")


## End-to-end runtime

Preprocessing is measured separately from analysis so device-specific filtering choices remain visible.

In [ ]:
analyzer = PyHEARTS(sampling_rate=FS, species="human")

t0 = time.perf_counter()
filtered = analyzer.preprocess_signal(
    ECG,
    highpass_cutoff=0.5,
    lowpass_cutoff=50.0,
    filter_order=4,
    notch_frequency=50.0,
    quality_factor=30.0,
)
preprocess_s = time.perf_counter() - t0

t0 = time.perf_counter()
features, cycles = analyzer.analyze_ecg(filtered)
analyze_s = time.perf_counter() - t0

pd.DataFrame(
    [
        {"stage": "preprocess", "seconds": preprocess_s},
        {"stage": "analyze", "seconds": analyze_s},
        {"stage": "total", "seconds": preprocess_s + analyze_s},
    ]
).assign(realtime_multiple=lambda x: DURATION_S / x["seconds"])

print(f"cycles={len(features)}; finite T={features['T_global_center_idx'].notna().sum()}")

## Morphology versus record-T timing

Estimate stage cost by timing the full analyzer, then timing the record-T helper
alone on the same R detections. Morphology time is end-to-end minus record-T.


In [ ]:
from pyhearts.core import analyzer as analyzer_mod

analyzer = PyHEARTS(sampling_rate=FS, species="human")

t0 = time.perf_counter()
features, cycles = analyzer.analyze_ecg(filtered)
end_to_end_s = time.perf_counter() - t0

r_values = pd.to_numeric(features["R_global_center_idx"], errors="coerce").to_numpy(dtype=float)
t0 = time.perf_counter()
pairs, stats = analyzer_mod.detect_record_t(
    filtered,
    r_values,
    FS,
    analyzer._t_cfg,
)
_ = analyzer_mod.merge_record_t(features.drop(columns=["T_gaussian_global_center_idx", "t_source"], errors="ignore"), pairs)
record_t_s = time.perf_counter() - t0
morphology_s = max(0.0, end_to_end_s - record_t_s)

print(pd.Series({
    "end_to_end_s": end_to_end_s,
    "morphology_est_s": morphology_s,
    "record_t_s": record_t_s,
    "record_t_fraction": record_t_s / end_to_end_s if end_to_end_s else float("nan"),
    "n_beats": len(features),
    "template_valid": stats.get("template_valid"),
}).to_string())


## Python hotspots

`cProfile` identifies the functions with the largest cumulative runtime.

In [ ]:
profiler = cProfile.Profile()
analyzer = PyHEARTS(sampling_rate=FS, species="human")
profiler.enable()
analyzer.analyze_ecg(filtered)
profiler.disable()

stream = io.StringIO()
pstats.Stats(profiler, stream=stream).strip_dirs().sort_stats("cumulative").print_stats(30)
print(stream.getvalue())